In [ ]:
import pandas as pd

# Load the training dataset
train_df = pd.read_csv('/content/train.csv')

# Display the first 5 rows of the DataFrame
display(train_df.head())

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [ ]:
# Display a concise summary of the DataFrame, including data types and non-null values
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          77299 non-null  int64  
 1   geohash        77299 non-null  object 
 2   day            77299 non-null  int64  
 3   timestamp      77299 non-null  object 
 4   demand         77299 non-null  float64
 5   RoadType       76699 non-null  object 
 6   NumberofLanes  77299 non-null  int64  
 7   LargeVehicles  77299 non-null  object 
 8   Landmarks      77299 non-null  object 
 9   Temperature    74804 non-null  float64
 10  Weather        76502 non-null  object 
dtypes: float64(2), int64(3), object(6)
memory usage: 6.5+ MB


From the `info()` output, we can observe the data types of each column and check for non-null counts. Next, let's explicitly check for missing values.

In [ ]:
# Check for missing values
missing_values = train_df.isnull().sum()

# Display columns with missing values and their counts
print("Missing values per column:")
print(missing_values[missing_values > 0])

Missing values per column:
RoadType        600
Temperature    2495
Weather         797
dtype: int64


Based on the missing value analysis, we will proceed with the following preprocessing steps:
1. Drop the 'Index' column as it's a unique identifier and not useful for modeling.
2. Convert the 'timestamp' column to a datetime object.
3. Impute missing values in 'RoadType' and 'Weather' with their respective modes (most frequent values).
4. Impute missing values in 'Temperature' with its mean.

In [ ]:
# Drop the 'Index' column, ignore error if it doesn't exist
train_df = train_df.drop('Index', axis=1, errors='ignore')

# Check if 'timestamp' column exists before trying to extract hour/minute and drop it
if 'timestamp' in train_df.columns:
    train_df['hour'] = train_df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    train_df['minute'] = train_df['timestamp'].apply(lambda x: int(x.split(':')[1]))
    train_df = train_df.drop('timestamp', axis=1)

# Impute 'RoadType' and 'Weather' with mode, only if columns exist
for col in ['RoadType', 'Weather']:
    if col in train_df.columns:
        mode_value = train_df[col].mode()[0]
        train_df[col] = train_df[col].fillna(mode_value) # Removed inplace=True

# Impute 'Temperature' with mean, only if the column exists and has missing values
if 'Temperature' in train_df.columns and train_df['Temperature'].isnull().any():
    mean_temp = train_df['Temperature'].mean()
    train_df['Temperature'] = train_df['Temperature'].fillna(mean_temp) # Removed inplace=True

# Display info and missing values again to confirm imputation
print("\nDataFrame Info after Preprocessing:")
train_df.info()

print("\nMissing values after Preprocessing:")
# Only print missing values for columns that are actually in the dataframe and still have NaNs
missing_after_processing = train_df.isnull().sum()
print(missing_after_processing[missing_after_processing > 0])


DataFrame Info after Preprocessing:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Columns: 1261 entries, day to minute
dtypes: bool(1255), float64(2), int64(4)
memory usage: 96.1 MB

Missing values after Preprocessing:
Series([], dtype: int64)


With missing values addressed, we can now proceed with feature engineering and encoding:
1. Extract time-based features (hour, day of week, day of year) from the 'timestamp' column.
2. Apply one-hot encoding to categorical features: 'geohash', 'RoadType', 'LargeVehicles', 'Landmarks', and 'Weather'.

In [ ]:
# Identify categorical columns for one-hot encoding
categorical_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

# Apply one-hot encoding
train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)

# Display the first few rows of the preprocessed DataFrame and its info
print("\nDataFrame Head after Feature Engineering and Encoding:")
display(train_df.head())

print("\nDataFrame Info after Feature Engineering and Encoding:")
train_df.info()

KeyError: "None of [Index(['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather'], dtype='object')] are in the [columns]"

After feature engineering and encoding, the next step is to scale the numerical features. This is important for many machine learning algorithms to perform optimally. We will use `StandardScaler` to transform the numerical columns, excluding the one-hot encoded features and the target variable (`demand`).

In [ ]:
from sklearn.preprocessing import StandardScaler

# Separate target variable (demand) from features
X = train_df.drop('demand', axis=1)
y = train_df['demand']

# Identify numerical columns to scale (excluding one-hot encoded columns)
# Columns with bool dtype are one-hot encoded and should not be scaled.
# Columns 'day', 'hour', 'minute', 'NumberofLanes', 'Temperature' are numerical features that need scaling.
numerical_cols_to_scale = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Initialize StandardScaler
scaler = StandardScaler()

# Apply scaling to the selected numerical columns
X[numerical_cols_to_scale] = scaler.fit_transform(X[numerical_cols_to_scale])

print("\nDataFrame Head after Scaling Numerical Features:")
display(X.head())

print("\nDataFrame Info after Scaling Numerical Features:")
X.info()


DataFrame Head after Scaling Numerical Features:


,day,NumberofLanes,Temperature,geohash_qp02yf,geohash_qp02yy,geohash_qp02yz,geohash_qp02z1,geohash_qp02z3,geohash_qp02z4,geohash_qp02z5,...,geohash_qp0dnn,RoadType_Residential,RoadType_Street,LargeVehicles_Not Allowed,Landmarks_Yes,Weather_Rainy,Weather_Snowy,Weather_Sunny,hour,minute
0,-0.336727,-1.121233,0.000000,False,False,False,True,False,False,False,...,False,False,False,True,False,False,False,False,-1.327501,-1.321463
1,-0.336727,1.089544,2.030268,False,False,False,False,False,False,False,...,False,True,False,False,True,False,False,True,-1.327501,-1.321463
2,-0.336727,-1.121233,1.314070,False,False,False,False,False,False,False,...,False,True,False,True,False,False,False,True,-1.327501,-1.321463
3,-0.336727,-1.121233,0.000000,False,False,False,False,False,False,False,...,False,True,False,True,False,True,False,False,-1.327501,-1.321463
4,-0.336727,-1.121233,-0.773710,False,False,False,False,False,False,False,...,False,True,False,True,False,True,False,False,-1.327501,-1.321463



DataFrame Info after Scaling Numerical Features:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Columns: 1260 entries, day to minute
dtypes: bool(1255), float64(5)
memory usage: 95.5 MB


With the data now fully preprocessed and scaled, the next step is to split it into training and validation sets. This is essential for training and evaluating machine learning models, helping to ensure that the model generalizes well to unseen data.

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and validation sets
# Using a 80/20 split, with a random state for reproducibility
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape: {y_val.shape}")

X_train shape: (61839, 1260)
X_val shape: (15460, 1260)
y_train shape: (61839,)
y_val shape: (15460,)


With the data now split into training and validation sets, we can proceed to train a machine learning model. We will use a `RandomForestRegressor` for this task.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Initialize the RandomForestRegressor model
# Using default parameters for simplicity, but these can be tuned
model = RandomForestRegressor(random_state=42, n_jobs=-1)

# Train the model on the training data
print("Training RandomForestRegressor...")
model.fit(X_train, y_train)
print("Training complete.")

# Make predictions on the validation set
y_pred = model.predict(X_val)

# Evaluate the model
mse = mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print(f"\nModel Evaluation on Validation Set:")
print(f"Mean Squared Error: {mse:.4f}")
print(f"R-squared: {r2:.4f}")

Training RandomForestRegressor...
Training complete.

Model Evaluation on Validation Set:
Mean Squared Error: 0.0012
R-squared: 0.9385


### Evaluating on Test Data and Generating Submission File

Now that the model is trained, let's preprocess the `test.csv` file, make predictions, and prepare a submission file.

In [ ]:
import pandas as pd

# Load the test dataset
test_df = pd.read_csv('/content/test.csv')

# Store original 'Index' for submission file
test_index = test_df['Index']

print("Test DataFrame head:")
display(test_df.head())

print("\nTest DataFrame info before preprocessing:")
test_df.info()

Test DataFrame head:


,Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,49,2:15,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02z9,49,2:15,Residential,1,Not Allowed,No,6.476213,Snowy
2,2,qp02yf,49,2:15,Residential,3,Allowed,Yes,22.318203,Sunny
3,3,qp02z6,49,2:15,Residential,2,Not Allowed,Yes,NaN,Rainy
4,4,qp02zd,49,2:15,Residential,1,Not Allowed,No,18.266162,Foggy



Test DataFrame info before preprocessing:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          41778 non-null  int64  
 1   geohash        41778 non-null  object 
 2   day            41778 non-null  int64  
 3   timestamp      41778 non-null  object 
 4   RoadType       41454 non-null  object 
 5   NumberofLanes  41778 non-null  int64  
 6   LargeVehicles  41778 non-null  object 
 7   Landmarks      41778 non-null  object 
 8   Temperature    40429 non-null  float64
 9   Weather        41347 non-null  object 
dtypes: float64(1), int64(3), object(6)
memory usage: 3.2+ MB


To ensure consistent preprocessing, we need to recalculate the modes for 'RoadType' and 'Weather' and the mean for 'Temperature' from the original training data. This is because the `train_df` in the current kernel state is already preprocessed and does not contain the original columns to derive these values reliably.

In [ ]:
# Load the original training dataset to get imputation values
original_train_df = pd.read_csv('/content/train.csv')

# Calculate mode for 'RoadType' and 'Weather' from original training data
mode_roadtype = original_train_df['RoadType'].mode()[0]
mode_weather = original_train_df['Weather'].mode()[0]

# Calculate mean for 'Temperature' from original training data
mean_temperature = original_train_df['Temperature'].mean()

print(f"Mode for RoadType (from training data): {mode_roadtype}")
print(f"Mode for Weather (from training data): {mode_weather}")
print(f"Mean for Temperature (from training data): {mean_temperature:.2f}")

Mode for RoadType (from training data): Residential
Mode for Weather (from training data): Sunny
Mean for Temperature (from training data): 16.41


Now, we apply the same preprocessing steps to the test data as were applied to the training data.

In [ ]:
# Preprocessing Test Data

# Drop the 'Index' column
test_df = test_df.drop('Index', axis=1, errors='ignore')

# Extract 'hour' and 'minute' from 'timestamp' and drop 'timestamp'
if 'timestamp' in test_df.columns:
    test_df['hour'] = test_df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    test_df['minute'] = test_df['timestamp'].apply(lambda x: int(x.split(':')[1]))
    test_df = test_df.drop('timestamp', axis=1)

# Impute missing values using modes/mean calculated from training data
if 'RoadType' in test_df.columns:
    test_df['RoadType'] = test_df['RoadType'].fillna(mode_roadtype)
if 'Weather' in test_df.columns:
    test_df['Weather'] = test_df['Weather'].fillna(mode_weather)
if 'Temperature' in test_df.columns:
    test_df['Temperature'] = test_df['Temperature'].fillna(mean_temperature)

# Identify categorical columns for one-hot encoding (same as training)
categorical_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

# Apply one-hot encoding
test_df_encoded = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

# Align columns with X_train (features used for training)
# Add missing columns (present in X_train but not in test_df_encoded) and fill with 0
missing_cols_in_test = set(X_train.columns) - set(test_df_encoded.columns)
for c in missing_cols_in_test:
    test_df_encoded[c] = 0

# Drop extra columns (present in test_df_encoded but not in X_train)
extra_cols_in_test = set(test_df_encoded.columns) - set(X_train.columns)
if extra_cols_in_test:
    test_df_encoded = test_df_encoded.drop(columns=list(extra_cols_in_test))

# Reorder columns to match X_train
X_test_processed = test_df_encoded[X_train.columns]

# Identify numerical columns to scale (same as training)
numerical_cols_to_scale = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Apply scaling to the selected numerical columns using the *fitted* scaler
X_test_processed[numerical_cols_to_scale] = scaler.transform(X_test_processed[numerical_cols_to_scale])

print("\nTest DataFrame head after all preprocessing:")
display(X_test_processed.head())

print("\nTest DataFrame info after all preprocessing:")
X_test_processed.info()


Test DataFrame head after all preprocessing:


,day,NumberofLanes,Temperature,geohash_qp02yf,geohash_qp02yy,geohash_qp02yz,geohash_qp02z1,geohash_qp02z3,geohash_qp02z4,geohash_qp02z5,...,geohash_qp0dnn,RoadType_Residential,RoadType_Street,LargeVehicles_Not Allowed,Landmarks_Yes,Weather_Rainy,Weather_Snowy,Weather_Sunny,hour,minute
0,2.969762,-1.121233,0.000000,False,0,False,True,False,False,False,...,0,True,False,True,False,False,False,True,-1.035786,-0.431441
1,2.969762,-1.121233,-1.371422,False,0,False,False,False,False,False,...,0,True,False,True,False,False,True,False,-1.035786,-0.431441
2,2.969762,1.089544,0.816688,True,0,False,False,False,False,False,...,0,True,False,False,True,False,False,True,-1.035786,-0.431441
3,2.969762,-0.015845,0.000000,False,0,False,False,False,False,False,...,0,True,False,True,True,True,False,False,-1.035786,-0.431441
4,2.969762,-1.121233,0.257016,False,0,False,False,False,False,False,...,0,True,False,True,False,False,False,False,-1.035786,-0.431441



Test DataFrame info after all preprocessing:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Columns: 1260 entries, day to minute
dtypes: bool(1186), float64(5), int64(69)
memory usage: 70.8 MB


Now, we will make predictions on the preprocessed test data and save them to a CSV file.

In [ ]:
# Make predictions on the preprocessed test data
test_predictions = model.predict(X_test_processed)

# Create a DataFrame for submission
submission_df = pd.DataFrame({'Index': test_index, 'demand': test_predictions})

# Save the submission file to CSV
submission_filename = 'test_predictions.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"Predictions saved to {submission_filename}")
print("\nSubmission DataFrame head:")
display(submission_df.head())

Predictions saved to test_predictions.csv

Submission DataFrame head:


,Index,demand
0,0,0.040832
1,1,0.009881
2,2,0.008534
3,3,0.010350
4,4,0.049499


You can download the `test_predictions.csv` file from your Colab environment.

### Hyperparameter Tuning for RandomForestRegressor

To improve the model's performance, we will perform hyperparameter tuning using `GridSearchCV`. This will help us find the optimal combination of parameters for our `RandomForestRegressor`.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

# Define the parameter grid to search
# Starting with a smaller grid to keep computation manageable.
# This can be expanded for a more exhaustive search.
param_grid = {
    'n_estimators': [100, 200, 300], # Number of trees in the forest
    'max_features': ['sqrt', 'log2'], # Number of features to consider when looking for the best split
    'max_depth': [10, 20, None], # Maximum number of levels in tree
    'min_samples_split': [2, 5], # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2] # Minimum number of samples required to be at a leaf node
}

# Initialize the RandomForestRegressor
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

# Initialize GridSearchCV
# verbose=2 will show progress during the search
# cv=3 for 3-fold cross-validation
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2, scoring='r2')

print("Starting GridSearchCV for RandomForestRegressor...")
# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)
print("GridSearchCV complete.")

Starting GridSearchCV for RandomForestRegressor...
Fitting 3 folds for each of 72 candidates, totalling 216 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


GridSearchCV complete.


In [ ]:
# Get the best parameters and best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"Best parameters found: {best_params}")
print(f"Best R2 score from cross-validation: {best_score:.4f}")

# Train a new model with the best parameters
best_rf_model = grid_search.best_estimator_

# Evaluate the new best model on the validation set
y_pred_tuned = best_rf_model.predict(X_val)
mse_tuned = mean_squared_error(y_val, y_pred_tuned)
r2_tuned = r2_score(y_val, y_pred_tuned)

print(f"\nModel Evaluation on Validation Set (after tuning):")
print(f"Mean Squared Error: {mse_tuned:.4f}")
print(f"R-squared: {r2_tuned:.4f}")

Best parameters found: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best R2 score from cross-validation: 0.9033

Model Evaluation on Validation Set (after tuning):
Mean Squared Error: 0.0017
R-squared: 0.9140


After hyperparameter tuning, we have a potentially improved model. The next step is to evaluate this `best_rf_model` on the test data and generate a new submission file to see if the R2 score and ranking have improved.

### Making Predictions with the Tuned Model and Generating New Submission

In [ ]:
# Make predictions on the preprocessed test data using the tuned model
tuned_test_predictions = best_rf_model.predict(X_test_processed)

# Create a DataFrame for the new submission
tuned_submission_df = pd.DataFrame({'Index': test_index, 'demand': tuned_test_predictions})

# Save the new submission file to CSV
tuned_submission_filename = 'tuned_test_predictions.csv'
tuned_submission_df.to_csv(tuned_submission_filename, index=False)

print(f"Tuned predictions saved to {tuned_submission_filename}")
print("\nNew Submission DataFrame head:")
display(tuned_submission_df.head())

Tuned predictions saved to tuned_test_predictions.csv

New Submission DataFrame head:


,Index,demand
0,0,0.036569
1,1,0.025458
2,2,0.026464
3,3,0.025364
4,4,0.047281


The `tuned_test_predictions.csv` file has been generated. Please download and evaluate this new submission. If the R2 score is still not meeting the 95% target, we might need to explore further strategies such as more extensive hyperparameter tuning, advanced feature engineering, or trying different models like XGBoost or LightGBM, or even stacking/ensembling methods. We could also consider more sophisticated outlier handling or a deeper analysis of specific feature impacts.

### Making Predictions with the Tuned Model and Generating New Submission

In [ ]:
# Make predictions on the preprocessed test data using the tuned model
tuned_test_predictions = best_rf_model.predict(X_test_processed)

# Create a DataFrame for the new submission
tuned_submission_df = pd.DataFrame({'Index': test_index, 'demand': tuned_test_predictions})

# Save the new submission file to CSV
tuned_submission_filename = 'tuned_test_predictions.csv'
tuned_submission_df.to_csv(tuned_submission_filename, index=False)

print(f"Tuned predictions saved to {tuned_submission_filename}")
print("\nNew Submission DataFrame head:")
display(tuned_submission_df.head())

Tuned predictions saved to tuned_test_predictions.csv

New Submission DataFrame head:


,Index,demand
0,0,0.036569
1,1,0.025458
2,2,0.026464
3,3,0.025364
4,4,0.047281


The `tuned_test_predictions.csv` file has been generated. Please download and evaluate this new submission. If the R2 score is still not meeting the 95% target, we might need to explore further strategies such as more extensive hyperparameter tuning, advanced feature engineering, or trying different models like XGBoost or LightGBM, or even stacking/ensembling methods. We could also consider more sophisticated outlier handling or a deeper analysis of specific feature impacts.

# Task
The main task is to develop and refine a machine learning model to predict traffic demand. This involves comprehensive data preprocessing, feature engineering, model training, hyperparameter tuning, and evaluation. The goal is to achieve an R2 score of 95% or higher on the test data. The process will involve loading and cleaning the data, creating new features, applying one-hot encoding and scaling, training a Random Forest Regressor and an XGBoost Regressor, tuning their hyperparameters, making predictions on test data, and generating submission files. The task will iterate on improving model performance based on evaluation metrics.

## Enhanced Feature Engineering & Data Preparation

### Subtask:
Reload the original training data and apply enhanced feature engineering, including deriving 'day_of_week' and 'is_weekend'. Re-apply all previous preprocessing steps (missing value imputation, one-hot encoding, scaling) to prepare the data for a new model.


**Reasoning**:
Reloading the original `train.csv` dataset into a new DataFrame as per the first instruction.



In [ ]:
import pandas as pd

# Reload the original training dataset into a new DataFrame
train_df_enhanced = pd.read_csv('/content/train.csv')

# Display the first few rows to confirm successful loading
print("Reloaded original training DataFrame head:")
display(train_df_enhanced.head())


Reloaded original training DataFrame head:


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


**Reasoning**:
I'm performing the enhanced feature engineering and re-applying initial preprocessing steps, including dropping the 'Index' column, converting 'timestamp' to datetime, extracting new time features, and handling missing values using modes and mean from the original training data. This sets up the `train_df_enhanced` for further processing.



In [ ]:
# Drop the 'Index' column as it's a unique identifier and not useful for modeling.
train_df_enhanced = train_df_enhanced.drop('Index', axis=1, errors='ignore')

# Convert 'timestamp' to datetime for feature extraction
# First, combine 'day' and 'timestamp' to create a full datetime object
# Assuming all days belong to the same 'month' and 'year' for simplicity in datetime conversion
# Let's assume a dummy year and month, as only relative time matters here for day_of_week and is_weekend
train_df_enhanced['datetime'] = pd.to_datetime(train_df_enhanced['timestamp'], format='%H:%M').dt.time
# Create a dummy date for 'day' to be able to extract day of week and weekend information
# We can use a reference date and add the 'day' column as days offset
reference_date = pd.to_datetime('2023-01-01') # A Monday
train_df_enhanced['full_datetime'] = train_df_enhanced.apply(lambda row: reference_date + pd.Timedelta(days=row['day']) + pd.Timedelta(hours=row['datetime'].hour, minutes=row['datetime'].minute), axis=1)

# Extract new time-based features
train_df_enhanced['hour'] = train_df_enhanced['full_datetime'].dt.hour
train_df_enhanced['minute'] = train_df_enhanced['full_datetime'].dt.minute
train_df_enhanced['day_of_week'] = train_df_enhanced['full_datetime'].dt.dayofweek # Monday=0, Sunday=6
train_df_enhanced['is_weekend'] = (train_df_enhanced['full_datetime'].dt.dayofweek >= 5).astype(int) # 1 for weekend, 0 for weekday

# Drop original 'timestamp' and 'full_datetime' column as they are no longer needed
train_df_enhanced = train_df_enhanced.drop(['timestamp', 'datetime', 'full_datetime'], axis=1)

# Calculate modes for 'RoadType' and 'Weather' from the original data
mode_roadtype = train_df_enhanced['RoadType'].mode()[0]
mode_weather = train_df_enhanced['Weather'].mode()[0]

# Calculate mean for 'Temperature' from the original data
mean_temperature = train_df_enhanced['Temperature'].mean()

# Impute missing values using the calculated modes and mean
train_df_enhanced['RoadType'] = train_df_enhanced['RoadType'].fillna(mode_roadtype)
train_df_enhanced['Weather'] = train_df_enhanced['Weather'].fillna(mode_weather)
train_df_enhanced['Temperature'] = train_df_enhanced['Temperature'].fillna(mean_temperature)

# Display info and missing values again to confirm imputation and new features
print("\nDataFrame Info after Enhanced Feature Engineering and Imputation:")
train_df_enhanced.info()

print("\nMissing values after Enhanced Feature Engineering and Imputation:")
missing_after_enhanced_processing = train_df_enhanced.isnull().sum()
print(missing_after_enhanced_processing[missing_after_enhanced_processing > 0])

print("\nDataFrame head after Enhanced Feature Engineering and Imputation:")
display(train_df_enhanced.head())


DataFrame Info after Enhanced Feature Engineering and Imputation:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   geohash        77299 non-null  object 
 1   day            77299 non-null  int64  
 2   demand         77299 non-null  float64
 3   RoadType       77299 non-null  object 
 4   NumberofLanes  77299 non-null  int64  
 5   LargeVehicles  77299 non-null  object 
 6   Landmarks      77299 non-null  object 
 7   Temperature    77299 non-null  float64
 8   Weather        77299 non-null  object 
 9   hour           77299 non-null  int32  
 10  minute         77299 non-null  int32  
 11  day_of_week    77299 non-null  int32  
 12  is_weekend     77299 non-null  int64  
dtypes: float64(2), int32(3), int64(3), object(5)
memory usage: 6.8+ MB

Missing values after Enhanced Feature Engineering and Imputation:
Series([], dtype: int64)


,geohash,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,hour,minute,day_of_week,is_weekend
0,qp02z1,48,0.048804,Residential,1,Not Allowed,No,16.405354,Sunny,0,0,5,1
1,qp02zt,48,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,0,0,5,1
2,qp08bj,48,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,0,0,5,1
3,qp08gt,48,0.003272,Residential,1,Not Allowed,No,16.405354,Rainy,0,0,5,1
4,qp02zq,48,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,0,0,5,1


**Reasoning**:
Performing one-hot encoding on specified categorical columns and then scaling the numerical features to prepare the data for modeling, ensuring consistency with previous preprocessing steps.



In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Identify categorical columns for one-hot encoding
categorical_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

# Apply one-hot encoding to train_df_enhanced
train_df_encoded = pd.get_dummies(train_df_enhanced, columns=categorical_cols, drop_first=True)

# Separate target variable (demand) from features
X = train_df_encoded.drop('demand', axis=1)
y = train_df_encoded['demand']

# Identify numerical columns to scale
# Exclude boolean (one-hot encoded) columns and other non-numerical types
numerical_cols_to_scale = X.select_dtypes(include=['int64', 'float64', 'int32']).columns.tolist()

# Initialize StandardScaler
scaler = StandardScaler()

# Apply scaling to the selected numerical columns
X[numerical_cols_to_scale] = scaler.fit_transform(X[numerical_cols_to_scale])

# Split the data into training and validation sets
X_train_enhanced, X_val_enhanced, y_train_enhanced, y_val_enhanced = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nDataFrame Head after Enhanced Feature Engineering, Encoding, and Scaling:")
display(X_train_enhanced.head())

print("\nDataFrame Info after Enhanced Feature Engineering, Encoding, and Scaling:")
X_train_enhanced.info()

print("\nX_train_enhanced shape:", X_train_enhanced.shape)
print("X_val_enhanced shape:", X_val_enhanced.shape)
print("y_train_enhanced shape:", y_train_enhanced.shape)
print("y_val_enhanced shape:", y_val_enhanced.shape)



DataFrame Head after Enhanced Feature Engineering, Encoding, and Scaling:


,day,NumberofLanes,Temperature,hour,minute,day_of_week,is_weekend,geohash_qp02yf,geohash_qp02yy,geohash_qp02yz,...,geohash_qp0dnh,geohash_qp0dnj,geohash_qp0dnn,RoadType_Residential,RoadType_Street,LargeVehicles_Not Allowed,Landmarks_Yes,Weather_Rainy,Weather_Snowy,Weather_Sunny
43522,-0.336727,-0.015845,-0.469473,0.422791,-0.431441,-0.336727,0.0,False,False,False,...,False,False,False,True,False,True,True,True,False,False
49351,-0.336727,-1.121233,-0.081891,0.714506,-1.321463,-0.336727,0.0,False,False,False,...,False,False,False,False,True,True,True,False,False,False
3693,-0.336727,-0.015845,-0.133973,-1.181643,-1.321463,-0.336727,0.0,False,False,False,...,False,False,False,True,False,True,True,False,False,False
39415,-0.336727,1.089544,0.357444,0.276933,-1.321463,-0.336727,0.0,False,False,False,...,False,False,False,True,False,False,True,False,False,False
9900,-0.336727,1.089544,-0.096070,-0.889928,-1.321463,-0.336727,0.0,False,False,False,...,False,False,False,True,False,False,True,False,False,False



DataFrame Info after Enhanced Feature Engineering, Encoding, and Scaling:
<class 'pandas.core.frame.DataFrame'>
Index: 61839 entries, 43522 to 15795
Columns: 1262 entries, day to Weather_Sunny
dtypes: bool(1255), float64(7)
memory usage: 77.8 MB

X_train_enhanced shape: (61839, 1262)
X_val_enhanced shape: (15460, 1262)
y_train_enhanced shape: (61839,)
y_val_enhanced shape: (15460,)


## Train and Evaluate XGBoost Model

### Subtask:
Train an XGBoost Regressor, a powerful gradient boosting model, on the newly engineered training data. Perform an initial evaluation on the validation set to establish a baseline before tuning.


**Reasoning**:
I will import the necessary libraries (`XGBRegressor`, `mean_squared_error`, `r2_score`), initialize an `XGBRegressor` with specified parameters, train the model on the enhanced training data (`X_train_enhanced`, `y_train_enhanced`), make predictions on the enhanced validation data (`X_val_enhanced`), and then calculate and print the Mean Squared Error (MSE) and R-squared (R2) scores to evaluate the model's performance.



In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

# Initialize XGBRegressor
# Using default parameters for simplicity, but these can be tuned later.
# random_state for reproducibility
# n_jobs=-1 to use all available CPU cores
model_xgb = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1)

print("Training XGBoost Regressor...")
# Train the model on the enhanced training data
model_xgb.fit(X_train_enhanced, y_train_enhanced)
print("Training complete.")

# Make predictions on the enhanced validation set
y_pred_xgb = model_xgb.predict(X_val_enhanced)

# Evaluate the model
mse_xgb = mean_squared_error(y_val_enhanced, y_pred_xgb)
r2_xgb = r2_score(y_val_enhanced, y_pred_xgb)

print(f"\nModel Evaluation on Validation Set (XGBoost baseline):")
print(f"Mean Squared Error: {mse_xgb:.4f}")
print(f"R-squared: {r2_xgb:.4f}")

Training XGBoost Regressor...
Training complete.

Model Evaluation on Validation Set (XGBoost baseline):
Mean Squared Error: 0.0023
R-squared: 0.8863


## Hyperparameter Tuning for XGBoost

### Subtask:
Conduct a comprehensive hyperparameter tuning for the XGBoost model using RandomizedSearchCV to find the optimal combination of parameters that maximize the R2 score.


**Reasoning**:
I will import the necessary classes for hyperparameter tuning and XGBoost, define the parameter distributions for RandomizedSearchCV, initialize the XGBoost Regressor, and then configure and run RandomizedSearchCV on the enhanced training data. Finally, I will print the best parameters and the R2 score from the search, and evaluate the best model on the validation set.



In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
from scipy.stats import uniform, randint
from sklearn.metrics import mean_squared_error, r2_score

# Define the parameter distribution grid for RandomizedSearchCV
# Using a dictionary where keys are parameter names and values are distributions or lists of values
param_distributions = {
    'n_estimators': randint(100, 500), # Number of boosting rounds
    'learning_rate': uniform(0.01, 0.2), # Step size shrinkage to prevent overfitting
    'max_depth': randint(3, 10), # Maximum depth of a tree
    'subsample': uniform(0.6, 0.4), # Subsample ratio of the training instance
    'colsample_bytree': uniform(0.6, 0.4), # Subsample ratio of columns when constructing each tree
    'gamma': uniform(0, 0.2), # Minimum loss reduction required to make a further partition on a leaf node
    'reg_alpha': uniform(0, 0.5), # L1 regularization term on weights
    'reg_lambda': uniform(0, 0.5) # L2 regularization term on weights
}

# Initialize XGBRegressor with a fixed random_state
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1)

# Initialize RandomizedSearchCV
# n_iter: Number of parameter settings that are sampled. (Adjust based on computational resources)
# cv: Number of folds for cross-validation
# scoring: Metric to optimize (R2 score)
# verbose: Controls the verbosity: the higher, the more messages.
# n_jobs: Number of jobs to run in parallel (-1 means using all processors)
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_distributions,
    n_iter=50,  # Number of random combinations to try. Can increase for more exhaustive search.
    cv=3,       # 3-fold cross-validation
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=2
)

print("Starting RandomizedSearchCV for XGBoost...")
# Fit RandomizedSearchCV to the enhanced training data
random_search.fit(X_train_enhanced, y_train_enhanced)
print("RandomizedSearchCV complete.")

# Get the best parameters and best R2 score
best_params_xgb = random_search.best_params_
best_score_xgb = random_search.best_score_

print(f"\nBest parameters found for XGBoost: {best_params_xgb}")
print(f"Best R2 score from cross-validation for XGBoost: {best_score_xgb:.4f}")

# Evaluate the best tuned XGBoost model on the enhanced validation set
best_xgb_model = random_search.best_estimator_
y_pred_xgb_tuned = best_xgb_model.predict(X_val_enhanced)

mse_xgb_tuned = mean_squared_error(y_val_enhanced, y_pred_xgb_tuned)
r2_xgb_tuned = r2_score(y_val_enhanced, y_pred_xgb_tuned)

print(f"\nModel Evaluation on Validation Set (Tuned XGBoost):")
print(f"Mean Squared Error: {mse_xgb_tuned:.4f}")
print(f"R-squared: {r2_xgb_tuned:.4f}")

Starting RandomizedSearchCV for XGBoost...
Fitting 3 folds for each of 50 candidates, totalling 150 fits
RandomizedSearchCV complete.

Best parameters found for XGBoost: {'colsample_bytree': np.float64(0.9193380499938204), 'gamma': np.float64(0.030143508793085895), 'learning_rate': np.float64(0.11163975534814373), 'max_depth': 7, 'n_estimators': 358, 'reg_alpha': np.float64(0.2954464715941209), 'reg_lambda': np.float64(0.3387821809211412), 'subsample': np.float64(0.6066351315711425)}
Best R2 score from cross-validation for XGBoost: 0.8809

Model Evaluation on Validation Set (Tuned XGBoost):
Mean Squared Error: 0.0022
R-squared: 0.8915


## Predict on Test Data with Tuned XGBoost

### Subtask:
Apply the exact same enhanced preprocessing and feature engineering steps to the test dataset. Use the best-tuned XGBoost model to make predictions and generate a new submission file for evaluation.


**Reasoning**:
This step reloads the test data, stores the original index, and applies the initial enhanced feature engineering and missing value imputation, mirroring the preprocessing applied to the training data. This ensures consistency before one-hot encoding and scaling.



In [ ]:
import pandas as pd

# Reload the original test dataset
test_df_enhanced = pd.read_csv('/content/test.csv')

# Store original 'Index' for submission file
test_index = test_df_enhanced['Index']

# Drop the 'Index' column
test_df_enhanced = test_df_enhanced.drop('Index', axis=1, errors='ignore')

# Convert 'timestamp' to datetime for feature extraction
# First, combine 'day' and 'timestamp' to create a full datetime object
# Assuming all days belong to the same 'month' and 'year' for simplicity in datetime conversion
# Use the same reference date as for training data
reference_date = pd.to_datetime('2023-01-01') # A Monday
test_df_enhanced['datetime'] = pd.to_datetime(test_df_enhanced['timestamp'], format='%H:%M').dt.time
test_df_enhanced['full_datetime'] = test_df_enhanced.apply(lambda row: reference_date + pd.Timedelta(days=row['day']) + pd.Timedelta(hours=row['datetime'].hour, minutes=row['datetime'].minute), axis=1)

# Extract new time-based features
test_df_enhanced['hour'] = test_df_enhanced['full_datetime'].dt.hour
test_df_enhanced['minute'] = test_df_enhanced['full_datetime'].dt.minute
test_df_enhanced['day_of_week'] = test_df_enhanced['full_datetime'].dt.dayofweek # Monday=0, Sunday=6
test_df_enhanced['is_weekend'] = (test_df_enhanced['full_datetime'].dt.dayofweek >= 5).astype(int) # 1 for weekend, 0 for weekday

# Drop original 'timestamp', 'datetime', and 'full_datetime' columns
test_df_enhanced = test_df_enhanced.drop(['timestamp', 'datetime', 'full_datetime'], axis=1)

# Impute missing values using modes/mean calculated from training data (global variables)
# mode_roadtype, mode_weather, mean_temperature are already available from previous steps
if 'RoadType' in test_df_enhanced.columns:
    test_df_enhanced['RoadType'] = test_df_enhanced['RoadType'].fillna(mode_roadtype)
if 'Weather' in test_df_enhanced.columns:
    test_df_enhanced['Weather'] = test_df_enhanced['Weather'].fillna(mode_weather)
if 'Temperature' in test_df_enhanced.columns:
    test_df_enhanced['Temperature'] = test_df_enhanced['Temperature'].fillna(mean_temperature)

print("Test DataFrame head after enhanced feature engineering and imputation:")
display(test_df_enhanced.head())

print("\nTest DataFrame info after enhanced feature engineering and imputation:")
test_df_enhanced.info()

print("\nMissing values in test_df_enhanced after processing:")
print(test_df_enhanced.isnull().sum()[test_df_enhanced.isnull().sum() > 0])

Test DataFrame head after enhanced feature engineering and imputation:


,geohash,day,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,hour,minute,day_of_week,is_weekend
0,qp02z1,49,Residential,1,Not Allowed,No,16.405354,Sunny,2,15,6,1
1,qp02z9,49,Residential,1,Not Allowed,No,6.476213,Snowy,2,15,6,1
2,qp02yf,49,Residential,3,Allowed,Yes,22.318203,Sunny,2,15,6,1
3,qp02z6,49,Residential,2,Not Allowed,Yes,16.405354,Rainy,2,15,6,1
4,qp02zd,49,Residential,1,Not Allowed,No,18.266162,Foggy,2,15,6,1



Test DataFrame info after enhanced feature engineering and imputation:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   geohash        41778 non-null  object 
 1   day            41778 non-null  int64  
 2   RoadType       41778 non-null  object 
 3   NumberofLanes  41778 non-null  int64  
 4   LargeVehicles  41778 non-null  object 
 5   Landmarks      41778 non-null  object 
 6   Temperature    41778 non-null  float64
 7   Weather        41778 non-null  object 
 8   hour           41778 non-null  int32  
 9   minute         41778 non-null  int32  
 10  day_of_week    41778 non-null  int32  
 11  is_weekend     41778 non-null  int64  
dtypes: float64(1), int32(3), int64(3), object(5)
memory usage: 3.3+ MB

Missing values in test_df_enhanced after processing:
Series([], dtype: int64)


**Reasoning**:
I need to apply one-hot encoding and scaling to the test data to match the format of the training data (`X_train_enhanced`) on which the `best_xgb_model` was trained. This includes aligning columns and using the previously fitted `scaler`.



In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify categorical columns for one-hot encoding (same as training)
categorical_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

# Apply one-hot encoding to test_df_enhanced
test_df_encoded = pd.get_dummies(test_df_enhanced, columns=categorical_cols, drop_first=True)

# Align columns with X_train_enhanced (features used for training the tuned XGBoost model)
# Add missing columns (present in X_train_enhanced but not in test_df_encoded) and fill with 0
missing_cols_in_test = set(X_train_enhanced.columns) - set(test_df_encoded.columns)
for c in missing_cols_in_test:
    test_df_encoded[c] = 0

# Drop extra columns (present in test_df_encoded but not in X_train_enhanced)
extra_cols_in_test = set(test_df_encoded.columns) - set(X_train_enhanced.columns)
if extra_cols_in_test:
    test_df_encoded = test_df_encoded.drop(columns=list(extra_cols_in_test))

# Reorder columns to match X_train_enhanced
X_test_enhanced_processed = test_df_encoded[X_train_enhanced.columns]

# Identify numerical columns to scale (same as training)
# These columns were defined when scaling X_train_enhanced
numerical_cols_to_scale_test = X_test_enhanced_processed.select_dtypes(include=['int64', 'float64', 'int32']).columns.tolist()

# Apply scaling to the selected numerical columns using the *fitted* scaler from training data
X_test_enhanced_processed[numerical_cols_to_scale_test] = scaler.transform(X_test_enhanced_processed[numerical_cols_to_scale_test])

print("\nTest DataFrame head after all enhanced preprocessing, encoding, and scaling:")
display(X_test_enhanced_processed.head())

print("\nTest DataFrame info after all enhanced preprocessing, encoding, and scaling:")
X_test_enhanced_processed.info()

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- geohash_qp02yy
- geohash_qp03vd
- geohash_qp03y5
- geohash_qp03yn
- geohash_qp03yr
- ...


**Reasoning**:
The previous code failed because the list of numerical columns to scale in the test set (`numerical_cols_to_scale_test`) incorrectly included one-hot encoded features, leading to a mismatch with the columns the `scaler` was fitted on. I will use the `numerical_cols_to_scale` variable, which was correctly identified from the training data, to select only the true numerical columns for scaling in the test set.



In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify categorical columns for one-hot encoding (same as training)
categorical_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

# Apply one-hot encoding to test_df_enhanced
test_df_encoded = pd.get_dummies(test_df_enhanced, columns=categorical_cols, drop_first=True)

# Align columns with X_train_enhanced (features used for training the tuned XGBoost model)
# Add missing columns (present in X_train_enhanced but not in test_df_encoded) and fill with 0
missing_cols_in_test = set(X_train_enhanced.columns) - set(test_df_encoded.columns)
for c in missing_cols_in_test:
    test_df_encoded[c] = 0

# Drop extra columns (present in test_df_encoded but not in X_train_enhanced)
extra_cols_in_test = set(test_df_encoded.columns) - set(X_train_enhanced.columns)
if extra_cols_in_test:
    test_df_encoded = test_df_encoded.drop(columns=list(extra_cols_in_test))

# Reorder columns to match X_train_enhanced
X_test_enhanced_processed = test_df_encoded[X_train_enhanced.columns]

# Use the numerical_cols_to_scale list from the training data to ensure consistency
# numerical_cols_to_scale was defined when scaling X_train_enhanced and contains only true numerical features

# Apply scaling to the selected numerical columns using the *fitted* scaler from training data
X_test_enhanced_processed[numerical_cols_to_scale] = scaler.transform(X_test_enhanced_processed[numerical_cols_to_scale])

print("\nTest DataFrame head after all enhanced preprocessing, encoding, and scaling:")
display(X_test_enhanced_processed.head())

print("\nTest DataFrame info after all enhanced preprocessing, encoding, and scaling:")
X_test_enhanced_processed.info()


Test DataFrame head after all enhanced preprocessing, encoding, and scaling:


,day,NumberofLanes,Temperature,hour,minute,day_of_week,is_weekend,geohash_qp02yf,geohash_qp02yy,geohash_qp02yz,...,geohash_qp0dnh,geohash_qp0dnj,geohash_qp0dnn,RoadType_Residential,RoadType_Street,LargeVehicles_Not Allowed,Landmarks_Yes,Weather_Rainy,Weather_Snowy,Weather_Sunny
0,2.969762,-1.121233,0.000000,-1.035786,-0.431441,2.969762,0.0,False,0,False,...,0,False,0,True,False,True,False,False,False,True
1,2.969762,-1.121233,-1.371422,-1.035786,-0.431441,2.969762,0.0,False,0,False,...,0,False,0,True,False,True,False,False,True,False
2,2.969762,1.089544,0.816688,-1.035786,-0.431441,2.969762,0.0,True,0,False,...,0,False,0,True,False,False,True,False,False,True
3,2.969762,-0.015845,0.000000,-1.035786,-0.431441,2.969762,0.0,False,0,False,...,0,False,0,True,False,True,True,True,False,False
4,2.969762,-1.121233,0.257016,-1.035786,-0.431441,2.969762,0.0,False,0,False,...,0,False,0,True,False,True,False,False,False,False



Test DataFrame info after all enhanced preprocessing, encoding, and scaling:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Columns: 1262 entries, day to Weather_Sunny
dtypes: bool(1186), float64(7), int64(69)
memory usage: 71.5 MB


**Reasoning**:
The next step is to make predictions using the `best_xgb_model` on the `X_test_enhanced_processed` data and then create a submission DataFrame with the 'Index' and 'demand' columns, saving it to a CSV file.



In [ ]:
import pandas as pd

# Make predictions on the preprocessed test data using the best-tuned XGBoost model
tuned_xgb_test_predictions = best_xgb_model.predict(X_test_enhanced_processed)

# Create a DataFrame for the new submission
tuned_xgb_submission_df = pd.DataFrame({'Index': test_index, 'demand': tuned_xgb_test_predictions})

# Save the new submission file to CSV
tuned_xgb_submission_filename = 'tuned_xgb_predictions.csv'
tuned_xgb_submission_df.to_csv(tuned_xgb_submission_filename, index=False)

print(f"Tuned XGBoost predictions saved to {tuned_xgb_submission_filename}")
print("\nNew XGBoost Submission DataFrame head:")
display(tuned_xgb_submission_df.head())

Tuned XGBoost predictions saved to tuned_xgb_predictions.csv

New XGBoost Submission DataFrame head:


,Index,demand
0,0,0.055012
1,1,0.055012
2,2,0.055012
3,3,0.055012
4,4,0.055012


## Final Task

### Subtask:
Summarize the improvements achieved with the new model and feature engineering, and provide guidance for further steps if the target R2 score is not yet met.


## Enhanced Feature Engineering: Adding Cyclical Features

To better capture the periodic nature of 'hour' and 'day_of_week', we will apply sine and cosine transformations to these features. This helps the model understand that, for example, 23:00 is closer to 00:00 than to 12:00.

In [ ]:
import numpy as np
import pandas as pd

# Reload the original training dataset into a new DataFrame for enhanced features
train_df_enhanced_v2 = pd.read_csv('/content/train.csv')

# Drop the 'Index' column
train_df_enhanced_v2 = train_df_enhanced_v2.drop('Index', axis=1, errors='ignore')

# Convert 'timestamp' to datetime for feature extraction
# Create a dummy date for 'day' to be able to extract day of week and weekend information
reference_date = pd.to_datetime('2023-01-01') # A Monday
train_df_enhanced_v2['datetime_obj'] = pd.to_datetime(train_df_enhanced_v2['timestamp'], format='%H:%M').dt.time
train_df_enhanced_v2['full_datetime'] = train_df_enhanced_v2.apply(lambda row: reference_date + pd.Timedelta(days=row['day']) + pd.Timedelta(hours=row['datetime_obj'].hour, minutes=row['datetime_obj'].minute), axis=1)

# Extract new time-based features
train_df_enhanced_v2['hour'] = train_df_enhanced_v2['full_datetime'].dt.hour
train_df_enhanced_v2['minute'] = train_df_enhanced_v2['full_datetime'].dt.minute
train_df_enhanced_v2['day_of_week'] = train_df_enhanced_v2['full_datetime'].dt.dayofweek # Monday=0, Sunday=6
train_df_enhanced_v2['is_weekend'] = (train_df_enhanced_v2['full_datetime'].dt.dayofweek >= 5).astype(int) # 1 for weekend, 0 for weekday

# --- Add Cyclical Features ---
# Cyclical features for hour (24 hours)
train_df_enhanced_v2['hour_sin'] = np.sin(2 * np.pi * train_df_enhanced_v2['hour'] / 24)
train_df_enhanced_v2['hour_cos'] = np.cos(2 * np.pi * train_df_enhanced_v2['hour'] / 24)

# Cyclical features for day_of_week (7 days)
train_df_enhanced_v2['day_of_week_sin'] = np.sin(2 * np.pi * train_df_enhanced_v2['day_of_week'] / 7)
train_df_enhanced_v2['day_of_week_cos'] = np.cos(2 * np.pi * train_df_enhanced_v2['day_of_week'] / 7)
# --- End Cyclical Features ---

# Drop original 'timestamp', 'datetime_obj', 'full_datetime' and now also 'hour', 'day_of_week' (as we have cyclical) columns
train_df_enhanced_v2 = train_df_enhanced_v2.drop(['timestamp', 'datetime_obj', 'full_datetime', 'hour', 'day_of_week'], axis=1)

# Calculate modes for 'RoadType' and 'Weather' from the original data
mode_roadtype_v2 = train_df_enhanced_v2['RoadType'].mode()[0]
mode_weather_v2 = train_df_enhanced_v2['Weather'].mode()[0]

# Calculate mean for 'Temperature' from the original data
mean_temperature_v2 = train_df_enhanced_v2['Temperature'].mean()

# Impute missing values using the calculated modes and mean
train_df_enhanced_v2['RoadType'] = train_df_enhanced_v2['RoadType'].fillna(mode_roadtype_v2)
train_df_enhanced_v2['Weather'] = train_df_enhanced_v2['Weather'].fillna(mode_weather_v2)
train_df_enhanced_v2['Temperature'] = train_df_enhanced_v2['Temperature'].fillna(mean_temperature_v2)

print("\nDataFrame Info after Enhanced Feature Engineering and Imputation (v2):")
train_df_enhanced_v2.info()

print("\nMissing values after Enhanced Feature Engineering and Imputation (v2):")
missing_after_enhanced_processing_v2 = train_df_enhanced_v2.isnull().sum()
print(missing_after_enhanced_processing_v2[missing_after_enhanced_processing_v2 > 0])

print("\nDataFrame head after Enhanced Feature Engineering and Imputation (v2):")
display(train_df_enhanced_v2.head())


DataFrame Info after Enhanced Feature Engineering and Imputation (v2):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   geohash          77299 non-null  object 
 1   day              77299 non-null  int64  
 2   demand           77299 non-null  float64
 3   RoadType         77299 non-null  object 
 4   NumberofLanes    77299 non-null  int64  
 5   LargeVehicles    77299 non-null  object 
 6   Landmarks        77299 non-null  object 
 7   Temperature      77299 non-null  float64
 8   Weather          77299 non-null  object 
 9   minute           77299 non-null  int32  
 10  is_weekend       77299 non-null  int64  
 11  hour_sin         77299 non-null  float64
 12  hour_cos         77299 non-null  float64
 13  day_of_week_sin  77299 non-null  float64
 14  day_of_week_cos  77299 non-null  float64
dtypes: float64(6), int32(1), int64(3

,geohash,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,minute,is_weekend,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos
0,qp02z1,48,0.048804,Residential,1,Not Allowed,No,16.405354,Sunny,0,1,0.0,1.0,-0.974928,-0.222521
1,qp02zt,48,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,0,1,0.0,1.0,-0.974928,-0.222521
2,qp08bj,48,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,0,1,0.0,1.0,-0.974928,-0.222521
3,qp08gt,48,0.003272,Residential,1,Not Allowed,No,16.405354,Rainy,0,1,0.0,1.0,-0.974928,-0.222521
4,qp02zq,48,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,0,1,0.0,1.0,-0.974928,-0.222521


### One-Hot Encoding, Scaling, and Train-Validation Split (v2)

Now, we apply one-hot encoding to the categorical features and scale the numerical features. Then we split the data into training and validation sets for the refined model.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Identify categorical columns for one-hot encoding
categorical_cols_v2 = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

# Apply one-hot encoding to train_df_enhanced_v2
train_df_encoded_v2 = pd.get_dummies(train_df_enhanced_v2, columns=categorical_cols_v2, drop_first=True)

# Separate target variable (demand) from features
X_v2 = train_df_encoded_v2.drop('demand', axis=1)
y_v2 = train_df_encoded_v2['demand']

# Identify numerical columns to scale
numerical_cols_to_scale_v2 = X_v2.select_dtypes(include=['int64', 'float64', 'int32']).columns.tolist()

# Initialize StandardScaler for the new dataset
scaler_v2 = StandardScaler()

# Apply scaling to the selected numerical columns
X_v2[numerical_cols_to_scale_v2] = scaler_v2.fit_transform(X_v2[numerical_cols_to_scale_v2])

# Split the data into training and validation sets
X_train_enhanced_v2, X_val_enhanced_v2, y_train_enhanced_v2, y_val_enhanced_v2 = train_test_split(X_v2, y_v2, test_size=0.2, random_state=42)

print("\nDataFrame Head after Enhanced Feature Engineering, Encoding, and Scaling (v2):")
display(X_train_enhanced_v2.head())

print("\nDataFrame Info after Enhanced Feature Engineering, Encoding, and Scaling (v2):")
X_train_enhanced_v2.info()

print("\nX_train_enhanced_v2 shape:", X_train_enhanced_v2.shape)
print("X_val_enhanced_v2 shape:", X_val_enhanced_v2.shape)
print("y_train_enhanced_v2 shape:", y_train_enhanced_v2.shape)
print("y_val_enhanced_v2 shape:", y_val_enhanced_v2.shape)


DataFrame Head after Enhanced Feature Engineering, Encoding, and Scaling (v2):


,day,NumberofLanes,Temperature,minute,is_weekend,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,geohash_qp02yf,...,geohash_qp0dnh,geohash_qp0dnj,geohash_qp0dnn,RoadType_Residential,RoadType_Street,LargeVehicles_Not Allowed,Landmarks_Yes,Weather_Rainy,Weather_Snowy,Weather_Sunny
43522,-0.336727,-0.015845,-0.469473,-0.431441,0.0,-0.293426,-1.406618,-0.336727,-0.336727,False,...,False,False,False,True,False,True,True,True,False,False
49351,-0.336727,-1.121233,-0.081891,-1.321463,0.0,-1.095966,-1.229799,-0.336727,-0.336727,False,...,False,False,False,False,True,True,True,False,False,False
3693,-0.336727,-0.015845,-0.133973,-1.321463,0.0,0.121999,1.188007,-0.336727,-0.336727,False,...,False,False,False,True,False,True,True,False,False,False
39415,-0.336727,1.089544,0.357444,-1.321463,0.0,0.121999,-1.361647,-0.336727,-0.336727,False,...,False,False,False,True,False,False,True,False,False,False
9900,-0.336727,1.089544,-0.096070,-1.321463,0.0,0.841537,0.846418,-0.336727,-0.336727,False,...,False,False,False,True,False,False,True,False,False,False



DataFrame Info after Enhanced Feature Engineering, Encoding, and Scaling (v2):
<class 'pandas.core.frame.DataFrame'>
Index: 61839 entries, 43522 to 15795
Columns: 1264 entries, day to Weather_Sunny
dtypes: bool(1255), float64(9)
memory usage: 78.7 MB

X_train_enhanced_v2 shape: (61839, 1264)
X_val_enhanced_v2 shape: (15460, 1264)
y_train_enhanced_v2 shape: (61839,)
y_val_enhanced_v2 shape: (15460,)


## Train and Evaluate XGBoost Model (v2) with Cyclical Features

We will now train an XGBoost Regressor on the data with the new cyclical features and evaluate its baseline performance.

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

# Initialize XGBRegressor
model_xgb_v2 = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1)

print("Training XGBoost Regressor (v2)...")
# Train the model on the enhanced training data (v2)
model_xgb_v2.fit(X_train_enhanced_v2, y_train_enhanced_v2)
print("Training complete.")

# Make predictions on the enhanced validation set (v2)
y_pred_xgb_v2 = model_xgb_v2.predict(X_val_enhanced_v2)

# Evaluate the model
mse_xgb_v2 = mean_squared_error(y_val_enhanced_v2, y_pred_xgb_v2)
r2_xgb_v2 = r2_score(y_val_enhanced_v2, y_pred_xgb_v2)

print(f"\nModel Evaluation on Validation Set (XGBoost baseline v2):")
print(f"Mean Squared Error: {mse_xgb_v2:.4f}")
print(f"R-squared: {r2_xgb_v2:.4f}")

Training XGBoost Regressor (v2)...
Training complete.

Model Evaluation on Validation Set (XGBoost baseline v2):
Mean Squared Error: 0.0023
R-squared: 0.8845


## Hyperparameter Tuning for XGBoost (v2) with Cyclical Features

We will now conduct hyperparameter tuning for the XGBoost model using RandomizedSearchCV on the data that includes cyclical features.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
from scipy.stats import uniform, randint
from sklearn.metrics import mean_squared_error, r2_score

# Define the parameter distribution grid for RandomizedSearchCV
param_distributions_v2 = {
    'n_estimators': randint(100, 500),
    'learning_rate': uniform(0.01, 0.2),
    'max_depth': randint(3, 10),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0, 0.2),
    'reg_alpha': uniform(0, 0.5),
    'reg_lambda': uniform(0, 0.5)
}

# Initialize XGBRegressor with a fixed random_state
xgb_model_v2 = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1)

# Initialize RandomizedSearchCV
random_search_v2 = RandomizedSearchCV(
    estimator=xgb_model_v2,
    param_distributions=param_distributions_v2,
    n_iter=50,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=2
)

print("Starting RandomizedSearchCV for XGBoost (v2)...")
# Fit RandomizedSearchCV to the enhanced training data (v2)
random_search_v2.fit(X_train_enhanced_v2, y_train_enhanced_v2)
print("RandomizedSearchCV complete.")

# Get the best parameters and best R2 score
best_params_xgb_v2 = random_search_v2.best_params_
best_score_xgb_v2 = random_search_v2.best_score_

print(f"\nBest parameters found for XGBoost (v2): {best_params_xgb_v2}")
print(f"Best R2 score from cross-validation for XGBoost (v2): {best_score_xgb_v2:.4f}")

# Evaluate the best tuned XGBoost model on the enhanced validation set (v2)
best_xgb_model_v2 = random_search_v2.best_estimator_
y_pred_xgb_tuned_v2 = best_xgb_model_v2.predict(X_val_enhanced_v2)

mse_xgb_tuned_v2 = mean_squared_error(y_val_enhanced_v2, y_pred_xgb_tuned_v2)
r2_xgb_tuned_v2 = r2_score(y_val_enhanced_v2, y_pred_xgb_tuned_v2)

print(f"\nModel Evaluation on Validation Set (Tuned XGBoost v2):")
print(f"Mean Squared Error: {mse_xgb_tuned_v2:.4f}")
print(f"R-squared: {r2_xgb_tuned_v2:.4f}")

Starting RandomizedSearchCV for XGBoost (v2)...
Fitting 3 folds for each of 50 candidates, totalling 150 fits
RandomizedSearchCV complete.

Best parameters found for XGBoost (v2): {'colsample_bytree': np.float64(0.9193380499938204), 'gamma': np.float64(0.030143508793085895), 'learning_rate': np.float64(0.11163975534814373), 'max_depth': 7, 'n_estimators': 358, 'reg_alpha': np.float64(0.2954464715941209), 'reg_lambda': np.float64(0.3387821809211412), 'subsample': np.float64(0.6066351315711425)}
Best R2 score from cross-validation for XGBoost (v2): 0.8800

Model Evaluation on Validation Set (Tuned XGBoost v2):
Mean Squared Error: 0.0022
R-squared: 0.8900


## Predict on Test Data with Tuned XGBoost (v2) and Cyclical Features

Now, we apply the same enhanced preprocessing, including cyclical features, to the test dataset, use the best-tuned XGBoost model (v2) to make predictions, and generate a new submission file.

In [ ]:
import pandas as pd
import numpy as np

# Reload the original test dataset
test_df_enhanced_v2 = pd.read_csv('/content/test.csv')

# Store original 'Index' for submission file
test_index_v2 = test_df_enhanced_v2['Index']

# Drop the 'Index' column
test_df_enhanced_v2 = test_df_enhanced_v2.drop('Index', axis=1, errors='ignore')

# Convert 'timestamp' to datetime for feature extraction
# Create a dummy date for 'day' to be able to extract day of week and weekend information
reference_date = pd.to_datetime('2023-01-01') # A Monday
test_df_enhanced_v2['datetime_obj'] = pd.to_datetime(test_df_enhanced_v2['timestamp'], format='%H:%M').dt.time
test_df_enhanced_v2['full_datetime'] = test_df_enhanced_v2.apply(lambda row: reference_date + pd.Timedelta(days=row['day']) + pd.Timedelta(hours=row['datetime_obj'].hour, minutes=row['datetime_obj'].minute), axis=1)

# Extract new time-based features
test_df_enhanced_v2['hour'] = test_df_enhanced_v2['full_datetime'].dt.hour
test_df_enhanced_v2['minute'] = test_df_enhanced_v2['full_datetime'].dt.minute
test_df_enhanced_v2['day_of_week'] = test_df_enhanced_v2['full_datetime'].dt.dayofweek # Monday=0, Sunday=6
test_df_enhanced_v2['is_weekend'] = (test_df_enhanced_v2['full_datetime'].dt.dayofweek >= 5).astype(int) # 1 for weekend, 0 for weekday

# --- Add Cyclical Features ---
# Cyclical features for hour (24 hours)
test_df_enhanced_v2['hour_sin'] = np.sin(2 * np.pi * test_df_enhanced_v2['hour'] / 24)
test_df_enhanced_v2['hour_cos'] = np.cos(2 * np.pi * test_df_enhanced_v2['hour'] / 24)

# Cyclical features for day_of_week (7 days)
test_df_enhanced_v2['day_of_week_sin'] = np.sin(2 * np.pi * test_df_enhanced_v2['day_of_week'] / 7)
test_df_enhanced_v2['day_of_week_cos'] = np.cos(2 * np.pi * test_df_enhanced_v2['day_of_week'] / 7)
# --- End Cyclical Features ---

# Drop original 'timestamp', 'datetime_obj', 'full_datetime' and now also 'hour', 'day_of_week' (as we have cyclical) columns
test_df_enhanced_v2 = test_df_enhanced_v2.drop(['timestamp', 'datetime_obj', 'full_datetime', 'hour', 'day_of_week'], axis=1)

# Impute missing values using modes/mean calculated from training data (global variables from v2)
if 'RoadType' in test_df_enhanced_v2.columns:
    test_df_enhanced_v2['RoadType'] = test_df_enhanced_v2['RoadType'].fillna(mode_roadtype_v2)
if 'Weather' in test_df_enhanced_v2.columns:
    test_df_enhanced_v2['Weather'] = test_df_enhanced_v2['Weather'].fillna(mode_weather_v2)
if 'Temperature' in test_df_enhanced_v2.columns:
    test_df_enhanced_v2['Temperature'] = test_df_enhanced_v2['Temperature'].fillna(mean_temperature_v2)

print("Test DataFrame head after enhanced feature engineering and imputation (v2):")
display(test_df_enhanced_v2.head())

print("\nTest DataFrame info after enhanced feature engineering and imputation (v2):")
test_df_enhanced_v2.info()

print("\nMissing values in test_df_enhanced_v2 after processing:")
print(test_df_enhanced_v2.isnull().sum()[test_df_enhanced_v2.isnull().sum() > 0])

Test DataFrame head after enhanced feature engineering and imputation (v2):


,geohash,day,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,minute,is_weekend,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos
0,qp02z1,49,Residential,1,Not Allowed,No,16.405354,Sunny,15,1,0.5,0.866025,-0.781831,0.62349
1,qp02z9,49,Residential,1,Not Allowed,No,6.476213,Snowy,15,1,0.5,0.866025,-0.781831,0.62349
2,qp02yf,49,Residential,3,Allowed,Yes,22.318203,Sunny,15,1,0.5,0.866025,-0.781831,0.62349
3,qp02z6,49,Residential,2,Not Allowed,Yes,16.405354,Rainy,15,1,0.5,0.866025,-0.781831,0.62349
4,qp02zd,49,Residential,1,Not Allowed,No,18.266162,Foggy,15,1,0.5,0.866025,-0.781831,0.62349



Test DataFrame info after enhanced feature engineering and imputation (v2):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   geohash          41778 non-null  object 
 1   day              41778 non-null  int64  
 2   RoadType         41778 non-null  object 
 3   NumberofLanes    41778 non-null  int64  
 4   LargeVehicles    41778 non-null  object 
 5   Landmarks        41778 non-null  object 
 6   Temperature      41778 non-null  float64
 7   Weather          41778 non-null  object 
 8   minute           41778 non-null  int32  
 9   is_weekend       41778 non-null  int64  
 10  hour_sin         41778 non-null  float64
 11  hour_cos         41778 non-null  float64
 12  day_of_week_sin  41778 non-null  float64
 13  day_of_week_cos  41778 non-null  float64
dtypes: float64(5), int32(1), int64(3), object(5)
memory usage: 4.3+ MB

Missi

### One-Hot Encoding, Column Alignment, and Scaling for Test Data (v2)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify categorical columns for one-hot encoding (same as training v2)
categorical_cols_v2 = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

# Apply one-hot encoding to test_df_enhanced_v2
test_df_encoded_v2 = pd.get_dummies(test_df_enhanced_v2, columns=categorical_cols_v2, drop_first=True)

# Align columns with X_train_enhanced_v2 (features used for training the tuned XGBoost model v2)
# Add missing columns (present in X_train_enhanced_v2 but not in test_df_encoded_v2) and fill with 0
missing_cols_in_test_v2 = set(X_train_enhanced_v2.columns) - set(test_df_encoded_v2.columns)
for c in missing_cols_in_test_v2:
    test_df_encoded_v2[c] = 0

# Drop extra columns (present in test_df_encoded_v2 but not in X_train_enhanced_v2)
extra_cols_in_test_v2 = set(test_df_encoded_v2.columns) - set(X_train_enhanced_v2.columns)
if extra_cols_in_test_v2:
    test_df_encoded_v2 = test_df_encoded_v2.drop(columns=list(extra_cols_in_test_v2))

# Reorder columns to match X_train_enhanced_v2
X_test_enhanced_processed_v2 = test_df_encoded_v2[X_train_enhanced_v2.columns]

# Use the numerical_cols_to_scale_v2 list from the training data to ensure consistency
# numerical_cols_to_scale_v2 was defined when scaling X_train_enhanced_v2

# Apply scaling to the selected numerical columns using the *fitted* scaler from training data
X_test_enhanced_processed_v2[numerical_cols_to_scale_v2] = scaler_v2.transform(X_test_enhanced_processed_v2[numerical_cols_to_scale_v2])

print("\nTest DataFrame head after all enhanced preprocessing, encoding, and scaling (v2):")
display(X_test_enhanced_processed_v2.head())

print("\nTest DataFrame info after all enhanced preprocessing, encoding, and scaling (v2):")
X_test_enhanced_processed_v2.info()


Test DataFrame head after all enhanced preprocessing, encoding, and scaling (v2):


,day,NumberofLanes,Temperature,minute,is_weekend,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,geohash_qp02yf,...,geohash_qp0dnh,geohash_qp0dnj,geohash_qp0dnn,RoadType_Residential,RoadType_Street,LargeVehicles_Not Allowed,Landmarks_Yes,Weather_Rainy,Weather_Snowy,Weather_Sunny
0,2.969762,-1.121233,0.000000,-0.431441,0.0,0.509114,1.056159,2.969762,2.969762,False,...,0,False,0,True,False,True,False,False,False,True
1,2.969762,-1.121233,-1.371422,-0.431441,0.0,0.509114,1.056159,2.969762,2.969762,False,...,0,False,0,True,False,True,False,False,True,False
2,2.969762,1.089544,0.816688,-0.431441,0.0,0.509114,1.056159,2.969762,2.969762,True,...,0,False,0,True,False,False,True,False,False,True
3,2.969762,-0.015845,0.000000,-0.431441,0.0,0.509114,1.056159,2.969762,2.969762,False,...,0,False,0,True,False,True,True,True,False,False
4,2.969762,-1.121233,0.257016,-0.431441,0.0,0.509114,1.056159,2.969762,2.969762,False,...,0,False,0,True,False,True,False,False,False,False



Test DataFrame info after all enhanced preprocessing, encoding, and scaling (v2):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Columns: 1264 entries, day to Weather_Sunny
dtypes: bool(1186), float64(9), int64(69)
memory usage: 72.1 MB


### Generate Submission File with Tuned XGBoost Model (v2)

In [ ]:
import pandas as pd

# Make predictions on the preprocessed test data using the best-tuned XGBoost model (v2)
tuned_xgb_test_predictions_v2 = best_xgb_model_v2.predict(X_test_enhanced_processed_v2)

# Create a DataFrame for the new submission
tuned_xgb_submission_df_v2 = pd.DataFrame({'Index': test_index_v2, 'demand': tuned_xgb_test_predictions_v2})

# Save the new submission file to CSV
tuned_xgb_submission_filename_v2 = 'tuned_xgb_predictions_v2.csv'
tuned_xgb_submission_df_v2.to_csv(tuned_xgb_submission_filename_v2, index=False)

print(f"Tuned XGBoost predictions (v2) saved to {tuned_xgb_submission_filename_v2}")
print("\nNew XGBoost Submission DataFrame head (v2):")
display(tuned_xgb_submission_df_v2.head())

Tuned XGBoost predictions (v2) saved to tuned_xgb_predictions_v2.csv

New XGBoost Submission DataFrame head (v2):


,Index,demand
0,0,0.055145
1,1,0.055145
2,2,0.055145
3,3,0.055145
4,4,0.055145


## Summary of Improvements and Next Steps

We have now run the XGBoost model with enhanced feature engineering, specifically adding cyclical features for 'hour' and 'day_of_week'. Please evaluate the `tuned_xgb_predictions_v2.csv` file.

Here's a recap of the R2 scores observed:
*   **RandomForestRegressor (Initial):** 0.9385
*   **RandomForestRegressor (Tuned):** 0.9140
*   **XGBoost (Baseline):** 0.8863
*   **XGBoost (Tuned, original features):** 0.8915
*   **XGBoost (Baseline, v2 with cyclical features):** [R2 from latest run will be here]
*   **XGBoost (Tuned, v2 with cyclical features):** [R2 from latest run will be here]

**If the R2 score is still not meeting your target of 95% after this iteration, we can consider the following techniques:**

1.  **More Complex Feature Interactions:** Manually create interaction terms between important features. For example, `Temperature * RoadType_Residential`.
2.  **External Data:** If available and allowed by the hackathon rules, incorporating external data (e.g., public holidays, special events, more precise weather data) could provide significant boosts.
3.  **Ensembling / Stacking:** Combine predictions from multiple different models (e.g., RandomForest, XGBoost, LightGBM, CatBoost) to leverage their diverse strengths. This often leads to more robust predictions.
4.  **Anomaly Detection and Outlier Handling:** Deep dive into demand values that are extremely high or low, and decide whether to treat them as outliers (remove/cap) or as valid, albeit rare, events that the model should learn from.
5.  **Time Series Specific Models:** While XGBoost can handle time-series, pure time-series models like ARIMA, Prophet, or even sequence models (LSTMs) might capture temporal dependencies even better, especially if the 'day' feature represents a long continuous time series.
6.  **Advanced Hyperparameter Tuning:** Explore a wider range of parameters, or use more advanced techniques like Bayesian Optimization (e.g., using libraries like `Optuna` or `Hyperopt`) for more efficient search.
7.  **Error Analysis (Continued):** Continue to analyze where the model performs poorly. Are there specific `geohash` locations, times of day, or weather conditions where predictions are consistently off? This can guide further targeted feature engineering.

Which of these would you like to explore next?